In [15]:
#@markdown # <-- 📍
#@markdown 1. Ýttu á "Run"-hnappin til vinstri
#@markdown     - mælt er með T4 GPU runtime
#@markdown 2. `pip` villa kemur upp:
#@markdown     - Runtime --> **Restart session**
#@markdown     - ⚠️veldu ekki 🚫`Disconnect and delete runtime`🚫
#@markdown     - --> ýttu aftur á **fyrsta hnappinn**
#@markdown 3. Hladdu upp "Skynjun sérhljóða og samhljóða kaflar 1-10.pdf" ásamt 10 	hljóðupptökum
#@markdown     - námsefni úr námskeiðinu ÍSE054G "Tal og hlustun I", eru á Uglu
#@markdown     - ekki hefur verið beðið um leyfi til að deila með öðrum.
#@markdown     - ⚠️án skjala er hægt að fletta niður og horfa á dæmi, en ekki hlusta/æfa.
#@markdown 4. Smelltu á næsta hnappin, bíddu, ...


#@markdown 💥 Hljóðupptaka er svolítið skrítin -
#@markdown *ef það virkar* tekið er upp í 2.sek
#@markdown en ekkert sýnir að það er í gangi -
#@markdown óþægilegt að gera í colab!

# ### installs etc

# for pdf extraction
!pip -q install pdfplumber

# for audio word timestamps
!pip -q install git+https://github.com/m-bain/whisperx.git
!://github.com/m-bain/whisperx.git
!apt-get -y -qq install ffmpeg
!pip -q install "numpy<2.3"

#print("When you get a pip error, restart the runtime session (DO NOT DISCONNECT/DELETE)")
#print("and run these installs again, it won't error")
#print("Recommend to run on GPU")
#print("Hladdu upp Skynjun sérhljóða og samhljóða kaflar 1-10.pdf")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
/bin/bash: line 1: ://github.com/m-bain/whisperx.git: No such file or directory


In [16]:
#@markdown # <--
##@markdown ### imports etc

import io, base64
import numpy as np
import os
import re
import json
import glob
import random
import pdfplumber
import difflib
import subprocess
import tempfile
import whisperx
import uuid
import librosa
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display, Audio, Javascript
from pydub import AudioSegment
from google.colab import output


In [17]:
#@markdown # <--
##@markdown ### pdf extraction functions

# skip unwanted pdf items
def cleanup_lines(text):
    output = []
    for raw in text.splitlines():

        #whitespace
        s = " ".join(raw.split()).strip()
        if not s:
            continue

        #booktitle
        if s == "Skynjun sérhljóða og samhljóða":
            continue

        #pagenum
        if re.fullmatch(r"\d+", s):
            continue

        #xnote
        if s.startswith("1 V er skammstöfun"):
            continue

        output.append(s)
    return output


# parse 2 colum IPA headers for x marks
def parse_header_cols(line):
    line_items = re.findall(r"\[([^\]]+)\]", line)
    if len(line_items) != 2: # not this linetype
        return None

    target_sounds = []
    for sound in line_items:
        # repair minor irregular format
        sound = re.sub(r"\s+", "", sound)
        sound = sound.replace("(:)", "")
        sound = sound.replace(":", "")
        if sound == "I":
            sound = "ɪ"
        target_sounds.append(sound)

    return {
        "target_sounds": target_sounds,
        "length": "langt" if any( x in line for x in (":","(:)")) else "stutt",
        "raw": line_items,
    }



# parse student exercises, p1-46
def parse_exercises(textar):

    kaflar = []
    this_k = None
    this_æ = None
    this_ææ = None

    for síða in textar:
        for line in cleanup_lines(síða):

            # is this a new chapter? it says kafli
            chh = re.match( r"^(\d+)\.\s*kafli(?:[.\s–-]+(.*))?$", line)
            if chh:
                chn = int(chh.group(1))
                cht = (chh.group(2) or "").strip()
                this_k = {
                    "number": chn,
                    "title": cht,
                    "description_lines": [],
                    "exercises": []
                }
                # add new chapter to db & open new exercises
                kaflar.append(this_k)
                this_æ = None
                this_ææ = None
                continue

            # is this a new exercise-set? it says æfing
            aef = re.match(r"^(\d+)\.\s*æfing:\s*(.*)$",line)
            if aef:
                aen = int(aef.group(1))
                aei = aef.group(2).strip()
               ################################## #if this_k is None:
                #    raise ValueError(
                #        f"Found exercise outside a chapter: {line}"
                #    )
                this_æ = {
                    "number": aen,
                    "title": aei,
                    "sub_exercises": []
                }
                this_k["exercises"].append(
                    this_æ
                )
                this_ææ = None
                continue

            # is this a new sub-exercise set? the column-header-parser returns it
            header = parse_header_cols(line)
            if header is not None and this_æ is not None:
                this_ææ = {
                    "length": header["length"],
                    "target_sounds": header["target_sounds"],
                    "words": []
                }
                this_æ["sub_exercises"].append(this_ææ)
                continue

            # if we already a current chapter but dont have an exercise open yet
            #   the current place must be in a chapter description
            if (this_k is not None and this_æ is None):
                this_k["description_lines"].append(line)
                continue

            # if we get here then the line is a single word's line
            #  only info wanted from this half of the book is the blank-printed wordform
            if this_ææ is not None and "__" in line:
                first_token = line.split()[0]
                this_ææ["words"].append({
                    "blank": first_token
                })

    for kafli in kaflar:
        kafli["summary"] = " ".join(kafli.pop("description_lines")).strip()
    return kaflar


#parse answer key info
def parse_key(textar):

    kaflar = []
    this_k = None
    this_æ = None
    this_ææ = None

    for síða in textar:
       for line in cleanup_lines(síða):

            # is this a new chapter? it says kafli
            chh = re.match(r"^(\d+)\.\s*kafli\s*[–-]\s*svör$",line)
            if chh:
                chn = int(chh.group(1))
                this_k = {
                    "number": chn,
                    "exercises": []
                }
                kaflar.append(this_k)
                this_æ = None
                this_ææ = None
                continue

            # new exercise-set? says æfing
            aef = re.match(r"^(\d+)\.\s*æfing:\s*(.*)$",line)
            if aef:
                aen = int(aef.group(1))
                aei = aef.group(2).strip()
                this_æ = {
                    "number": aen,
                    "title": aei,
                    "sub_exercises": []
                }
                this_k["exercises"].append(this_æ)
                this_ææ = None
                continue


            # subset column header?
            header = parse_header_cols(line)
            if header is not None and this_æ is not None:
                this_ææ = {
                    "length": header["length"],
                    "target_sounds": header["target_sounds"],
                    "rows": []
                }
                this_æ["sub_exercises"].append(this_ææ)
                continue

            if this_ææ is None:
                continue
            tokens = line.split()
            if len(tokens) < 4:
                continue

            # if we get here it should be a row of answers for a single word
            word = tokens[0]
            final_word = tokens[-1] # identical to word, in the answer key
            soundcheck_cols = tokens[1:-1]
            finding_x = [i for i, token in enumerate(soundcheck_cols)
                if re.search(r"x", token, flags=re.IGNORECASE) ]
            #if len(finding_x) != 1:
            #    continue
            assert len(finding_x) == 1
            target_index = finding_x[0]
            if target_index not in (0, 1):
                raise ValueError(
                    f"Unexpected X position in answer row: {line}"
                )

            this_ææ["rows"].append({
                "word": word,
                "target_sound": this_ææ[
                    "target_sounds"
                ][target_index]
            })

    return kaflar



def validate_pdf(exercise_chapters,answer_chapters):
  if len(exercise_chapters) != len(answer_chapters):
    raise ValueError(f"{len(exercise_chapters)} ≠ {len(answer_chapters)}")

  for ex_chapter, ans_chapter in zip(exercise_chapters,answer_chapters):
    if ex_chapter["number"] != ans_chapter["number"]:
        raise ValueError("Kaflar: " f"{ex_chapter['number']} ≠ " f"{ans_chapter['number']}")
    if len(ex_chapter["exercises"]) != len(ans_chapter["exercises"]):
        raise ValueError(f"Æfingar, "f"{ex_chapter['number']}.kafli")

    for ex, ans in zip(ex_chapter["exercises"],ans_chapter["exercises"]):
        if ex["number"] != ans["number"]:
            raise ValueError(
                f"{ex_chapter['number']}.kafli, afing "
                f"{ex['number']} ≠ {ans['number']}"
            )
        if len(ex["sub_exercises"]) != 2:
            raise ValueError(
                f"fann ekki (bara) langar & stuttar útgáfur - "
                f"kafli {ex_chapter['number']}, "
                f"æfing {ex['number']}"
            )
        if len(ans["sub_exercises"]) != 2:
            raise ValueError(
                f"fann ekki (bara) langar & stuttar útgáfur - "
                f"kafli {ans_chapter['number']}, "
                f"æfing {ans['number']}, svör"
            )

        for ex_sub, ans_sub in zip(
            ex["sub_exercises"],
            ans["sub_exercises"]
        ):
            if len(ex_sub["words"]) != 6:
                raise ValueError(
                    f"Það voru ekki (bara) 6 orð: "
                    f"kafli {ex_chapter['number']}, "
                    f"æfing {ex['number']}, "
                    f"{ex_sub['length']}; "
                    f"fann {len(ex_sub['words'])} orð"
                )
            if len(ans_sub["rows"]) != 6:
                raise ValueError(
                    f"Það voru ekki (bara) 6 svör:"
                    f"kafli {ans_chapter['number']}, "
                    f"æfing {ans['number']}, "
                    f"{ans_sub['length']}; "
                    f"fann {len(ans_sub['rows'])} orð"
                )
            # Target sounds should agree.
            if ex_sub["target_sounds"] != ans_sub[
                "target_sounds"
            ]:
                raise ValueError(
                    f"eitthvað fór vitlaust"
                    f"kafli {ex_chapter['number']}, "
                    f"æfing {ex['number']}, "
                    f"{ex_sub['length']}"
                )

  print("allt í lagi")


def merge_bookhalfs(exercise_chapters, answer_chapters):
  merged_chapters = []
  for ex_chapter, ans_chapter in zip(exercise_chapters,answer_chapters):
      merged_exercises = []
      for ex, ans in zip(ex_chapter["exercises"],ans_chapter["exercises"]):
          merged_sub_exercises = []
          for ex_sub, ans_sub in zip(ex["sub_exercises"],ans["sub_exercises"]):
              merged_words = []
              for blank_row, answer_row in zip(ex_sub["words"],ans_sub["rows"]):
                  merged_words.append({
                    "spurning": blank_row["blank"],
                    "orð": answer_row["word"],
                    "svar": answer_row["target_sound"]
                    })
              merged_sub_exercises.append({
                "lengd": ex_sub["length"],
                "hljóð": ex_sub["target_sounds"],
                "dæmi": merged_words
                })
          merged_exercises.append({
            "n_æf": ex["number"],
            "hljóðskynjun": ex["title"],
            "æfingar": merged_sub_exercises
            })
      merged_chapters.append({
        "n_kaf": ex_chapter["number"],
        "nafn": ex_chapter["title"],
        "lýsing": ex_chapter["summary"],
        "kafli": merged_exercises
        })

  return merged_chapters


def write_a_json(jdict):
  # why not here i guess
  outp = "/content/skynjun_serhljoda_samhljoda.json"
  with open(outp,"w",encoding="utf-8") as handle:
    json.dump(jdict,handle,ensure_ascii=False,indent=2)
  print("json:", outp)
  return outp

def extract_pdf():
  with pdfplumber.open(PDF_PATH) as pdf:
    if len(pdf.pages) != 87:
        print(
            f"WExpected 87 pages, found {len(pdf.pages)}, did you upload the right book?"
        )
    page_texts = [page.extract_text() or "" for page in pdf.pages]

  exercise_chapters = parse_exercises(page_texts[:46])
  answer_chapters = parse_key(page_texts[46:])
  validate_pdf(exercise_chapters,answer_chapters)

  all_exercises = merge_bookhalfs(exercise_chapters,answer_chapters)
  jdict = {"bók": all_exercises}
  bokp = write_a_json(jdict)
  return jdict, bokp



In [19]:
#@markdown # <--
##@markdown ### slice audio functions

DEVICE = "cuda"  # use "cpu" if no GPU, but will be slow
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
WHISPER_MODEL = "language-and-voice-lab/whisper-large-icelandic-62640-steps-967h-ct2"
ALIGN_MODEL_NAME = "language-and-voice-lab/wav2vec2-large-xlsr-53-icelandic-ep30-967h"

_ICE_CHARS = "a-záðéíóúýþæö"
_clean = lambda w: re.sub(f"[^{_ICE_CHARS}]", "", w.lower())

_whisper_model = None
_align_model = None
_align_meta = None

# whisper for ASR, w2v2 for alignment to whisper's xcp
#  (will match it to the book transcript later)
def load_falign_models():
    global _whisper_model, _align_model, _align_meta
    if _whisper_model is None:
        _whisper_model = whisperx.load_model(
            WHISPER_MODEL, DEVICE, language="is", compute_type=COMPUTE_TYPE
        )
    if _align_model is None:
        _align_model, _align_meta = whisperx.load_align_model(
            language_code="is", device=DEVICE, model_name=ALIGN_MODEL_NAME
        )

# a couple provided audios are WMA so
def standardise_audio(audio_path):
    tmp_path = tempfile.mktemp(suffix=".wav")
    cmd = [
        "ffmpeg", "-y", "-i", audio_path,
        "-ac", "1", "-ar", "16000",
        "-f", "wav", tmp_path,
        "-loglevel", "error",
    ]
    subprocess.run(cmd, check=True)
    return tmp_path


# there's one audio file per chapter
# get a flattened copy of metadata for each word in it
def get_transcript_items(book, chapter):
    items = []
    for kafli in book["bók"]:
        if kafli["n_kaf"] != chapter:
            continue
        for hs in kafli["kafli"]:
            hs_id = hs["hljóðskynjun"]
            for læf in hs["æfingar"]:
                lengd_id = læf["lengd"]
                sounds = læf["hljóð"]  # e.g. ["a", "i"]
                for dæmi in læf["dæmi"]:
                    item = dict(dæmi)
                    item["hljóðskynjun"] = hs_id
                    item["lengd"] = lengd_id
                    item["hljóð"] = sounds
                    items.append(item)
    if items:
        return items
    else:
        raise ValueError(f"Vantar kafli {chapter}")


# sometimes asr squishes 2 words into 1, unsquish before forced alignment
def asr_matcher(token, gold_pieces, min_avg_ratio = 0.55):
    n, k = len(token), len(gold_pieces)
    if k == 0 or n == 0:
        return None
    NEG = -1.0
    # dp[i][j] = (score, split_start_positions) best splitting token[:i]
    # into j pieces matching gold_pieces[:j]
    dp = [[(NEG, None)] * (k + 1) for _ in range(n + 1)]
    dp[0][0] = (0.0, [])
    for i in range(1, n + 1):
        for j in range(1, min(k, i) + 1):
            best = (NEG, None)
            for p in range(j - 1, i):
                prev_score, prev_splits = dp[p][j - 1]
                if prev_score < 0:
                    continue
                piece = token[p:i]
                ratio = difflib.SequenceMatcher(None, piece, gold_pieces[j - 1]).ratio()
                score = prev_score + ratio
                if score > best[0]:
                    best = (score, prev_splits + [p])
            dp[i][j] = best
    final_score, splits = dp[n][k]
    if splits is None:
        return None
    avg = final_score / k
    if avg < min_avg_ratio:
        return None
    bounds = splits + [n]
    pieces = [token[bounds[m]:bounds[m + 1]] for m in range(k)]
    return pieces, avg


def unsquish_asr(asr_segments, gold_words, min_avg_ratio = 0.55,max_extra_words = 3):
    gold_norm = [_clean(w) for w in gold_words]

    flat = []
    seg_word_lists = []
    for si, seg in enumerate(asr_segments):
        words = seg["text"].split()
        seg_word_lists.append(words)
        for wi, w in enumerate(words):
            flat.append({"text": w, "clean": _clean(w), "seg": si, "pos": wi})

    flat_clean = [f["clean"] for f in flat]
    sm = difflib.SequenceMatcher(None, flat_clean, gold_norm, autojunk=False)

    repairs = {}
    n_repairs = 0
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag != "replace" or (i2 - i1) == (j2 - j1):
            continue

        block = flat[i1:i2]
        gold_block = gold_norm[j1:j2]
        gold_orig = gold_words[j1:j2]
        k = len(gold_block)
        n = len(block)

        best = None  # (avg, sub_list)
        for start in range(n):
            max_len = min(n - start, k + max_extra_words)
            for length in range(1, max_len + 1):
                sub = block[start:start + length]
                if len({f["seg"] for f in sub}) != 1:
                    continue  # window must sit inside a single whisper segment
                concat = "".join(f["clean"] for f in sub)
                split = asr_matcher(concat, gold_block, min_avg_ratio=0.0)
                if split is None:
                    continue
                _, avg = split
                if best is None or avg > best[0]:
                    best = (avg, sub)

        if best is None:
            continue
        avg, sub = best
        if avg < min_avg_ratio:
            print(f"[segmentation-repair] REJECTED best-window avg={avg:.2f}: "
                  f"'{' '.join(f['text'] for f in sub)}' vs {gold_orig}")
            continue

        seg_idx = sub[0]["seg"]
        start_pos, end_pos = sub[0]["pos"], sub[-1]["pos"] + 1
        repairs.setdefault(seg_idx, []).append((start_pos, end_pos, gold_orig))
        n_repairs += 1
        print(f"[segmentation-repair] '{' '.join(f['text'] for f in sub)}' "
              f"-> '{' '.join(gold_orig)}' (avg char-similarity {avg:.2f})")

    if n_repairs:
        print(f"[segmentation-repair] applied {n_repairs} fix(es) before forced alignment")

    new_segments = []
    for si, seg in enumerate(asr_segments):
        words = seg_word_lists[si]
        for start_pos, end_pos, replacement in sorted(
            repairs.get(si, []), key=lambda x: x[0], reverse=True
        ):
            words[start_pos:end_pos] = replacement
        new_seg = dict(seg)
        new_seg["text"] = " ".join(words)
        new_segments.append(new_seg)

    return new_segments


def align_chapter(audio_path,book_contents):
    load_falign_models()
    wav_path = standardise_audio(audio_path)
    fuzzy_threshold=0.5

    # gets chapter # 1-10 for the patterns in my current set of files...
    chapter = int(re.search(r"skynjun\s+(\d+)", audio_path, re.I).group(1))

    transcript_tasks = get_transcript_items(book_contents,chapter)
    transcript_words = [d["orð"] for d in transcript_tasks]
    transcript_norm = [_clean(w) for w in transcript_words]

    try:
        asr = _whisper_model.transcribe(wav_path, language="is")
        asr["segments"] = unsquish_asr(asr["segments"], transcript_words)
        aligned = whisperx.align(asr["segments"], _align_model, _align_meta, wav_path, DEVICE)
    finally:
        os.remove(wav_path)

    asr_words = aligned["word_segments"]  #{"word","start","end"}
    asr_norm = [_clean(w["word"]) for w in asr_words]


    # correspond the ordered exercises to the full asr output
    sm = difflib.SequenceMatcher(None, asr_norm, transcript_norm, autojunk=False)
    timings = [None] * len(transcript_words)   # {"start","end"} or None
    exact = [None] * len(transcript_words)     # True / False / None

    # try to match and fuzzy/interpolate
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            for k in range(i2 - i1):
                w = asr_words[i1 + k]
                timings[j1 + k] = {"start": w["start"], "end": w["end"]}
                exact[j1 + k] = True
            continue

        # if something exists in book but not exact match found in asr...
        next_min = i1
        for j in range(j1, j2):
            gt_tok = transcript_norm[j]
            candidates = list(range(next_min, i2))
            best_idx, best_ratio = None, 0.0
            for c in candidates:
                r = difflib.SequenceMatcher(None, gt_tok, asr_norm[c]).ratio()
                if r > best_ratio:
                    best_ratio, best_idx = r, c

            if best_idx is not None and best_ratio >= fuzzy_threshold:
                w = asr_words[best_idx]
                timings[j] = {"start": w["start"], "end": w["end"]}
                exact[j] = False
                next_min = best_idx + 1  # stay monotonic about it
                print(f"[near-match] chapter {chapter}: book word '{transcript_words[j]}' "
                      f"matched to ASR word '{asr_words[best_idx]['word'].strip()}' "
                      f"(similarity {best_ratio:.2f}) at {w['start']:.2f}-{w['end']:.2f}s")
            else:
                timings[j] = {"start": None, "end": None}
                exact[j] = None

    result = [
        {"word": w,
         "start": timings[j]["start"], "end": timings[j]["end"],
         "exact": exact[j],
         "spurning": transcript_tasks[j].get("spurning"),
         "svar": transcript_tasks[j].get("svar"),
         "original_target_sounds": transcript_tasks[j].get("hljóð"),
         "original_hljóðskynjun": transcript_tasks[j].get("hljóðskynjun"),
         "lengd": transcript_tasks[j].get("lengd"),
         "audio_path": audio_path,
         }
        for j, w in enumerate(transcript_words)
    ]

    n_near = sum(1 for e in exact if e is False)
    n_missing = sum(1 for e in exact if e is None)
    if n_near or n_missing:
        print(f"[warn] chapter {chapter}: {n_near} near-matched, {n_missing} unmatched, "
              f"out of {len(transcript_words)} words")

    return result




In [25]:
#@markdown # <--
##@markdown ### ui functions


js_code = "navigator.mediaDevices.getUserMedia({audio: true}).then(function(stream) { stream.getTracks().forEach(function(t) { t.stop(); }); });"

display(Javascript(js_code))


# caching bc custom sets can load from several chapters audios
_AUDIO_CACHE = {}

def get_cached_audio(path):
    if path not in _AUDIO_CACHE:
        _AUDIO_CACHE[path] = load_audio_playable(path)
    return _AUDIO_CACHE[path]


# javascript apparently
#  why do i have to do blob media
#  dont like this.
# so ok forcing colab to record user audio is possibly not great
def rec_user_audio(seconds=2):
    js = f"""
    (async () => {{
      const b2text = blob => new Promise(resolve => {{
        const reader = new FileReader()
        reader.onloadend = e => resolve(e.srcElement.result)
        reader.readAsDataURL(blob)
      }})
      const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }})
      const recorder = new MediaRecorder(stream)
      const chunks = []
      recorder.ondataavailable = e => chunks.push(e.data)
      const stopped = new Promise(resolve => {{
        recorder.onstop = async () => {{
          const blob = new Blob(chunks)
          resolve(await b2text(blob))
        }}
      }})
      recorder.start()
      await new Promise(r => setTimeout(r, {int(seconds * 1000)}))
      recorder.stop()
      const result = await stopped
      stream.getTracks().forEach(t => t.stop())
      return result
    }})()
    """
    data_url = output.eval_js(js)
    if data_url is None:
        raise RuntimeError(
            "Recording failed — check that microphone permission was granted "
            "in the browser (look for a blocked-permission icon in the address bar)."
        )
    _, encoded = data_url.split(",", 1)
    return base64.b64decode(encoded)


# turn audio file into sth that can hopefully
#  handle playback in most browsers
def load_audio_playable(audio_path):
    wav_path = standardise_audio(audio_path)
    return AudioSegment.from_wav(wav_path)



# np array of audio, enforce mono
def s2sample(segment):
    samples = np.array(segment.get_array_of_samples())
    if segment.channels == 2:
        samples = samples.reshape((-1, 2)).mean(axis=1)
    return samples.astype(np.float32), segment.frame_rate

# make a html component for audio player -- for colab
def sound_for_html(segment):
    samples, rate = s2sample(segment)
    return Audio(samples, rate=rate, autoplay=False)._repr_html_()


# basic colour spectrogram
#  p much just matplots default everything for specgram so far
def see_specgm(segment):
    samples, rate = s2sample(segment)
    fig, ax = plt.subplots(figsize=(5, 2.2))
    ax.specgram(samples, Fs=rate, cmap="viridis")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("freq (Hz)")
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    plt.close(fig)
    buf.seek(0)
    return buf.read()


# for the ebook experience
def get_chapter_meta(book, chapter):
    for kafli in book["bók"]:
        if kafli["n_kaf"] == chapter:
            return kafli.get("nafn"), kafli.get("lýsing")
    return None, None

# for resampling
def get_flat_meta(tmp_ald):
    wtokens = []
    for entry in tmp_ald.values():
        for w in entry["times"]:
            if w.get("start") is not None:
                wtokens.append(w)
    return wtokens


# setup interactive exercise for user
#  listen, answer, optionally record self,
#   submit for scores at end of exercise set.
class TaskPlayer:
    def __init__(self, word_entry, radio_options, show_spectrograms_toggle,
                 hint_word_toggle, audios_getter, pad_ms=150, record_seconds=2,
                 show_lengd=False):
        self.w = word_entry
        self.audio = audios_getter(self.w)
        self.audio_len_ms = len(self.audio)
        self.pad_ms = pad_ms
        self.record_seconds = record_seconds
        self.show_spectrograms_toggle = show_spectrograms_toggle
        self.hint_word_toggle = hint_word_toggle
        self.show_lengd = show_lengd
        self._recorded_clip = None

        self.provided_audio_box = widgets.HTML("")
        self.provided_spec_img = widgets.Image(format="png",
                                                layout=widgets.Layout(display="none", width="250px"))
        self.target_sound_radio = widgets.RadioButtons(options=radio_options, value=None, description="Hljóð:")
        self.lengd_radio = None
        lengd_row = []
        if self.show_lengd:
            self.lengd_radio = widgets.RadioButtons(
                options=[("Stutt", "stutt"), ("Lang", "langt")], value=None, description="Lengd:")
            lengd_row = [self.lengd_radio]

        self.answer_text = widgets.Textarea(description="Orð:", layout=widgets.Layout(width="200px"))

        self.record_button = widgets.Button(description="Röddin mín")
        self.record_button.on_click(self._on_record)

        self.recorded_audio_box = widgets.HTML("")
        self.recorded_spec_img = widgets.Image(format="png",
                                                layout=widgets.Layout(display="none", width="250px"))
        self.widget = widgets.VBox([
            widgets.HTML("<div style='height:30px'></div><hr style='border:1px solid #888; margin:0'><div style='height:10px'></div>"),
            widgets.HBox([self.provided_audio_box, self.provided_spec_img]),
            self.target_sound_radio,
            *lengd_row,
            self.answer_text,
            self.record_button,
            widgets.HBox([self.recorded_audio_box, self.recorded_spec_img]),
        ])

        self._render_provided()
        self.answer_text.value = (self.w.get("spurning") or "") if hint_word_toggle.value else ""

    def _clip(self):
        start_ms = max(0, int(self.w["start"] * 1000) - self.pad_ms)
        end_ms = min(self.audio_len_ms, int(self.w["end"] * 1000) + self.pad_ms)
        return self.audio[start_ms:end_ms]

    def _render_provided(self):
        if self.w["start"] is None:
            self.provided_audio_box.value = f"<p>'{self.w['word']}' has no timestamp — nothing to play.</p>"
            self.provided_spec_img.layout.display = "none"
            return
        self.provided_audio_box.value = sound_for_html(self._clip())
        self._refresh_spectrogram(self.provided_spec_img, self._clip())

    def _refresh_spectrogram(self, img_widget, clip):
        if self.show_spectrograms_toggle.value:
            img_widget.value = see_specgm(clip)
            img_widget.layout.display = None
        else:
            img_widget.layout.display = "none"

    def refresh_options(self):
        if self.w["start"] is not None:
            self._refresh_spectrogram(self.provided_spec_img, self._clip())
        if self._recorded_clip is not None:
            self._refresh_spectrogram(self.recorded_spec_img, self._recorded_clip)

    def refresh_hint(self):
        self.answer_text.value = (self.w.get("spurning") or "") if self.hint_word_toggle.value else ""

    def _on_record(self, _button):
        raw = rec_user_audio(self.record_seconds)
        clip = AudioSegment.from_file(io.BytesIO(raw))
        self._recorded_clip = clip
        self.recorded_audio_box.value = sound_for_html(clip)
        self._refresh_spectrogram(self.recorded_spec_img, clip)

    def check(self):
        correct_sound = self.w.get("svar")
        correct_word = self.w.get("orð", self.w.get("word"))
        sound_ok = self.target_sound_radio.value == f'[{correct_sound}]'
        word_ok = self.answer_text.value.strip().lower() == str(correct_word).strip().lower()
        result = {
            "word": self.w.get("word"),
            "sound_ok": sound_ok, "picked_sound": self.target_sound_radio.value,
            "correct_sound": correct_sound,
            "word_ok": word_ok, "typed_word": self.answer_text.value,
            "correct_word": correct_word,
            "show_lengd": self.show_lengd,
        }
        if self.show_lengd:
            correct_lengd = self.w.get("lengd")
            result["lengd_ok"] = self.lengd_radio.value == correct_lengd
            result["picked_lengd"] = self.lengd_radio.value
            result["correct_lengd"] = correct_lengd
        return result


# assume for now all practise sets come from the same audio file
#  e.g. making an ebook of the original book
#     but not necessarily picking your own
#      custom 2-4 vowel sets and remixing tokens etc
class PracticeAndCheck:

    def __init__(self, selected_words, audios_getter, pad_ms=150,
                 record_seconds=2, show_lengd=False):
        self.show_spectrograms = widgets.Checkbox(value=False,
                                                  description="Sýna hljóðrófsrit")
        self.hint_word = widgets.Checkbox(value=True, description="Gefa vísbendingar")

        radio_options = sorted({f'[{w.get("svar")}]' for w in selected_words if w.get("svar")})

        self.slots = [
            TaskPlayer(w, radio_options, self.show_spectrograms, self.hint_word,
                       audios_getter, pad_ms=pad_ms,
                       record_seconds=record_seconds,
                       show_lengd=show_lengd)
            for w in selected_words
        ]

        self.feedback = widgets.HTML("")
        self.check_button = widgets.Button(description="Check answers", button_style="info")
        self.check_button.on_click(self._on_check)

        self.widget = widgets.VBox([
            widgets.HBox([self.show_spectrograms, self.hint_word]),
            *[slot.widget for slot in self.slots],
            self.feedback,
            self.check_button,
        ])
        display(self.widget)

        self.show_spectrograms.observe(self._on_toggle_spectrograms, names="value")
        self.hint_word.observe(self._on_toggle_hint, names="value")

    def _on_toggle_spectrograms(self, change):
        if change["name"] != "value":
            return
        for slot in self.slots:
            slot.refresh_options()

    def _on_toggle_hint(self, change):
        if change["name"] != "value":
            return
        for slot in self.slots:
            slot.refresh_hint()

    def _on_check(self, _button):
        lines = []
        for r in (slot.check() for slot in self.slots):
            lines.append(f"<b>{r['word']}</b>")
            lines.append(
                "✅ Hljóð: rétt!" if r["sound_ok"] else
                f"❌ Hljóð: þú valdir '{r['picked_sound']}', "
                f"en rétta svarið er ['{r['correct_sound']}']"
            )
            if r.get("show_lengd"):
                lines.append(
                    "✅ Lengd: rétt!" if r["lengd_ok"] else
                    f"❌ Lengd: þú valdir '{r['picked_lengd']}', "
                    f"en rétta svarið er '{r['correct_lengd']}'"
                )
            lines.append(
                "✅ Orð: rétt!" if r["word_ok"] else
                f"❌ Orð: þú skrifaði '{r['typed_word']}', "
                f"en orðið er '{r['correct_word']}'"
            )
            lines.append("<hr>")
        self.feedback.value = "<br>".join(lines)




def structured_selector(book_contents, tmp_ald, pad_ms=150, record_seconds=2):
    kafli_dropdown = widgets.Dropdown(
        options=sorted(tmp_ald.keys()), description="Kafli:")
    hs_dropdown = widgets.Dropdown(options=[], description="Hljóðskynjun:")
    lengd_dropdown = widgets.Dropdown(options=[], description="Lengd:")
    go_button = widgets.Button(description="Byrja", button_style="success")

    meta_html = widgets.HTML("")
    out = widgets.Output()

    def on_kafli_change(change):
        if change["name"] != "value":
            return
        chapter = change["new"]
        result = tmp_ald[chapter]["times"]

        nafn, lysing = get_chapter_meta(book_contents, chapter)
        meta_html.value = f"<h4>{nafn or ''}</h4><p>{lysing or ''}</p>"

        hs_values = sorted({w["original_hljóðskynjun"] for w in result if w.get("original_hljóðskynjun") is not None})
        hs_dropdown.options = hs_values
        if hs_values:
            hs_dropdown.index = 0  # force a definite selection + refresh

    def on_hs_change(change):
        if change["name"] != "value":
            return
        chapter = kafli_dropdown.value
        result = tmp_ald[chapter]["times"]
        hs = change["new"]
        lengd_values = sorted({
            w["lengd"] for w in result
            if w.get("original_hljóðskynjun") == hs and w.get("lengd") is not None
        })
        lengd_dropdown.options = lengd_values
        if lengd_values:
            lengd_dropdown.index = 0

    def on_go(_button):
        with out:
            out.clear_output()
            chapter = kafli_dropdown.value
            hs = hs_dropdown.value
            lengd = lengd_dropdown.value
            result = tmp_ald[chapter]["times"]
            chosen = [
                w for w in result
                if w.get("original_hljóðskynjun") == hs and w.get("lengd") == lengd
            ]
            if not chosen:
                print("Dæmi eru ekki til.")
                return
            audio = load_audio_playable(tmp_ald[chapter]["audio"])
            PracticeAndCheck(chosen, audios_getter=lambda w, a=audio: a,
                              pad_ms=pad_ms, record_seconds=record_seconds)

    kafli_dropdown.observe(on_kafli_change, names="value")
    hs_dropdown.observe(on_hs_change, names="value")
    go_button.on_click(on_go)

    display(meta_html, kafli_dropdown, hs_dropdown, lengd_dropdown, go_button, out)

    on_kafli_change({"name": "value", "new": kafli_dropdown.value})


def custom_task_maker(tmp_ald, words_per=3, pad_ms=150, record_seconds=2):
    sfrom = get_flat_meta(tmp_ald)
    all_sounds = sorted({w["svar"] for w in sfrom if w.get("svar")})

    sound_select = widgets.SelectMultiple(
        options=[(f'[{s}]',s) for s in sorted(all_sounds)],
        description="Hljóð:",
        layout=widgets.Layout(width="150px", height="200px"))
    lengd_select = widgets.SelectMultiple(
        options=[("Stutt", "stutt"), ("Löng", "langt")],
        value=("stutt",),
        style={"description_width": "200px"},
        description="Lengd sérhljóða:",
        layout=widgets.Layout(width="275px", height="60px"))
    go_button = widgets.Button(layout=widgets.Layout(width='max-content'),
                               description="Búa til æfingu (bíddu aðeins!)", button_style="success")
    out = widgets.Output()
    display(sound_select, lengd_select, go_button, out)

    def sample_unique(sound, lengd, n, used_texts):
        candidates = [
            w for w in sfrom
            if w.get("svar") == sound and w.get("lengd") == lengd
            and _clean(w["word"]) not in used_texts
        ]
        random.shuffle(candidates)
        chosen, seen = [], set()
        for w in candidates:
            key = _clean(w["word"])
            if key in seen:
                continue
            chosen.append(w)
            seen.add(key)
            if len(chosen) == n:
                break
        return chosen

    def on_go(_button):
        with out:
            out.clear_output()
            sounds = list(sound_select.value)
            lengdir = list(lengd_select.value)
            n_sounds, n_lens = len(sounds),len(lengdir)
            combos = (
                (2 <= n_sounds <= 4 and n_lens == 1) or
                (1 <= n_sounds <= 3 and n_lens == 2)
            )

            if not combos:
                print("Veldu 2-4 hljóð og 1 lengd, "
                      "eða 1-3 hljóð og 2 lengdir.")
                return

            selected = []
            for sound in sounds:
                for lengd in lengdir:
                    used_texts = {_clean(w["word"]) for w in selected}
                    chosen = sample_unique(sound, lengd, words_per, used_texts)
                    if len(chosen) < words_per:
                        print(f"[warn] fann aðeins {len(chosen)}/{words_per} orð -"
                              f" hljóð '[{sound}]', lengd '{lengd}'")
                    selected.extend(chosen)

            random.shuffle(selected)

            PracticeAndCheck(
                selected,
                audios_getter=lambda w: get_cached_audio(w["audio_path"]),
                pad_ms=pad_ms, record_seconds=record_seconds,
                show_lengd=(n_lens == 2),
            )

    go_button.on_click(on_go)

<IPython.core.display.Javascript object>

In [21]:
#@markdown ### <-- ⚠️ krefst pdf + hljóðskráa
##@markdown ### run pdf extraction


PDF_PATH = glob.glob("/content/Skynjun*.pdf")
if PDF_PATH:
  PDF_PATH = Path(PDF_PATH[0])
  print("Bók", PDF_PATH)
else:
  raise Exception("Hladdu upp Skynjun sérhljóða og samhljóða kaflar 1-10.pdf")

bokc, bokf = extract_pdf()

Bók /content/Skynjun sérhljóða og samhljóða kaflar 1-10.pdf
allt í lagi
json: /content/skynjun_serhljoda_samhljoda.json


In [22]:
#@markdown ### <-- ⚠️ krefst pdf + hljóðskráa
#@markdown (kaffihlé)
##@markdown ### run audio sample extraction

#can be .wav or WMA so like be careful not to upload other stuff this might catch
audios_paths= glob.glob("/content/[Ss]kynjun*.[wW]*")


TMP_ALD = {}
for af in audios_paths:
  chn = int(re.search(r"skynjun\s+(\d+)", af, re.I).group(1))
  aln = align_chapter(af,bokc)
  TMP_ALD[chn] = {"audio":af, "times": aln}


2026-09-14 04:56:05 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

[segmentation-repair] 'usskoss koddu' -> 'uss koss komdu' (avg char-similarity 0.93)
[segmentation-repair] 'úffsi' -> 'úff ufsi' (avg char-similarity 0.83)
[segmentation-repair] 'kestur' -> 'kæstur' (avg char-similarity 0.83)
[segmentation-repair] 'reykjaröku' -> 'reykja ruku' (avg char-similarity 0.88)
[segmentation-repair] 'lumaháma káthlutur gullháll' -> 'luma háma kát hlutur gull háll' (avg char-similarity 1.00)
[segmentation-repair] 'skrumfuni' -> 'skrum funi' (avg char-similarity 1.00)
[segmentation-repair] applied 6 fix(es) before forced alignment
[near-match] chapter 7: book word 'kúga' matched to ASR word 'kúa' (similarity 0.86) at 42.85-43.49s
[warn] chapter 7: 1 near-matched, 0 unmatched, out of 84 words
[segmentation-repair] 'tonntækla' -> 'tonn tækla' (avg char-similarity 1.00)
[segmentation-repair] 'fosskássa' -> 'foss kássa' (avg char-similarity 1.00)
[segmentation-repair] 'fornkostur fóstu' -> 'forn kostur fórstu' (avg char-similarity 0.97)
[segmentation-repair] applied

In [23]:
#@markdown ### <--
#@markdown # ⭐ Skynjun sérhljóða og samhljóða kaflar 1-10

structured_selector(bokc, TMP_ALD)

HTML(value='')

Dropdown(description='Kafli:', options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10), value=1)

Dropdown(description='Hljóðskynjun:', options=(), value=None)

Dropdown(description='Lengd:', options=(), value=None)

Button(button_style='success', description='Byrja', style=ButtonStyle())

Output()

In [26]:
#@markdown ### <--
#@markdown # 🌟 Sérsniðnar æfingar

#@markdown Veldu 2-4 hljóð og 1 lengd, eða 1-3 hljóð og 2 lengdir.



custom_task_maker(TMP_ALD)

SelectMultiple(description='Hljóð:', layout=Layout(height='200px', width='150px'), options=(('[a]', 'a'), ('[a…

SelectMultiple(description='Lengd sérhljóða:', index=(0,), layout=Layout(height='60px', width='275px'), option…

Button(button_style='success', description='Búa til æfingu (bíddu aðeins!)', layout=Layout(width='max-content'…

Output()